# Capitolo 3 — Un problema più difficile con lo stesso modello: Fashion-MNIST (§ 3.10)
Stesso MLP, stesso ciclo: cambia solo la riga che carica i dati.

In [ ]:
import sys; sys.path.insert(0, "..")
from utils import fissa_seme
import dati
import numpy as np
import matplotlib.pyplot as plt

import torch
from torch import nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
fissa_seme(42)

def mlp():
    return nn.Sequential(nn.Flatten(), nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 10))

perdita_fn = nn.CrossEntropyLoss()

def valuta(modello, dl):
    modello.eval(); corretti, totale, perdita_tot = 0, 0, 0.0
    with torch.no_grad():
        for xb, yb in dl:
            uscita = modello(xb)
            perdita_tot += perdita_fn(uscita, yb).item() * len(yb)
            corretti += (uscita.argmax(dim=1) == yb).sum().item(); totale += len(yb)
    return perdita_tot / totale, corretti / totale

def addestra(modello, ottimizzatore, train_ds, test_dl, epoche=5, batch_size=64, stampa=True):
    dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    storia = {"train": [], "test": [], "acc": [], "passi": []}
    for epoca in range(epoche):
        modello.train(); somma, n = 0.0, 0
        for i, (xb, yb) in enumerate(dl):
            perdita = perdita_fn(modello(xb), yb)
            ottimizzatore.zero_grad(); perdita.backward(); ottimizzatore.step()
            somma += perdita.item() * len(yb); n += len(yb)
            if i % 10 == 0: storia["passi"].append(perdita.item())
        pt, at = valuta(modello, test_dl)
        storia["train"].append(somma / n); storia["test"].append(pt); storia["acc"].append(at)
        if stampa: print(f"Epoca {epoca+1}: perdita train {somma/n:.4f} | perdita test {pt:.4f} | accuratezza test {at:.2%}")
    return storia

trasforma = transforms.ToTensor()
train_ds = datasets.FashionMNIST("../data", train=True,  download=True, transform=trasforma)
test_ds  = datasets.FashionMNIST("../data", train=False, download=True, transform=trasforma)
test_dl = DataLoader(test_ds, batch_size=1000)
classi = train_ds.classes; print(classi)

In [ ]:
fig, assi = plt.subplots(2, 10, figsize=(12, 3))
for i, ax in enumerate(assi.flat):
    img, et = train_ds[i]; ax.imshow(img[0], cmap="gray"); ax.set_title(classi[et], fontsize=7); ax.axis("off")
plt.show()

In [ ]:
fissa_seme(42)
modello = mlp()
storia = addestra(modello, torch.optim.Adam(modello.parameters(), lr=1e-3), train_ds, test_dl, epoche=5)

## La matrice di confusione dice dove il modello fatica

In [ ]:
from sklearn.metrics import confusion_matrix
modello.eval(); vere, previste = [], []
with torch.no_grad():
    for xb, yb in test_dl:
        vere += yb.tolist(); previste += modello(xb).argmax(1).tolist()
C = confusion_matrix(vere, previste)
Cn = C / C.sum(1, keepdims=True)
fig, ax = plt.subplots(figsize=(6, 5.5))
ax.imshow(Cn, cmap="Blues")
for i in range(10):
    for j in range(10):
        if i == j: ax.text(j, i, f"{Cn[i,i]:.0%}", ha="center", va="center", fontsize=7, color="white")
        elif C[i, j] >= 20: ax.text(j, i, C[i, j], ha="center", va="center", fontsize=7)
ax.set_xticks(range(10)); ax.set_yticks(range(10)); ax.set_xticklabels(classi, rotation=90); ax.set_yticklabels(classi); ax.set_xlabel("prevista"); ax.set_ylabel("vera")
plt.show()
print("per classe:", {c: f"{Cn[i,i]:.1%}" for i, c in enumerate(classi)})